# discriminator-classifier-head — worked example 3: PatchGAN head reduced to a per-image score by mean over patches

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `discriminator-classifier-head`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

A PatchGAN head uses a `1x1 Conv2d(C, 1)` + `Sigmoid` to produce a `(B, 1, H, W)` map of per-patch real/fake probabilities. To compare against a scalar-head discriminator you often want one number per image, which you get by averaging the patch map over its spatial axes. The result is the mean fraction of patches the discriminator judged real.

## Worked solution

**Goal:** turn `x: (B, C, H, W)` into a per-patch probability map `(B, 1, H, W)` and a per-image scalar `(B,)`.

1. **1x1 conv to one channel.** `conv` is `nn.Conv2d(C, 1, kernel_size=1)`. A 1x1 conv is literally a per-pixel linear map across channels, so `conv(x)` is `(B, 1, H, W)` logits — one logit per spatial location.
2. **Sigmoid for the patch map.** `t.sigmoid(logits)` → `(B, 1, H, W)`, each value a patch's real probability.
3. **Aggregate to a scalar.** `reduce(patch_map, 'b c h w -> b', 'mean')` averages over the channel (size 1) and both spatial axes, yielding `(B,)`.

Why this works: PatchGAN's insight is that realism is a *local* property — each receptive-field patch can be judged independently, which sharpens texture detail. The 1x1 conv shares one classifier across all positions (weight-tying), so it is far cheaper than a flatten+Linear and resolution-independent. Mean-pooling the patch scores recovers a single overall verdict when you need one.

In [ ]:
import torch.nn as nn
class PatchMeanHead(nn.Module):
    def __init__(self, in_channels: int):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, 1, kernel_size=1)

    def forward(self, x: Tensor):
        patch_map = t.sigmoid(self.conv(x))                 # (B, 1, H, W)
        score = reduce(patch_map, 'b c h w -> b', 'mean')   # (B,)
        return patch_map, score

t.manual_seed(0)
B, C, H, W = 3, 12, 6, 6
head = PatchMeanHead(C)
x = t.randn(B, C, H, W)
patch_map, score = head(x)
print('patch_map shape:', tuple(patch_map.shape))
print('score shape:', tuple(score.shape))
print('all patches in (0,1):', bool(((patch_map > 0) & (patch_map < 1)).all()))
print('scores:', score.round(decimals=3).tolist())